# Day 3: Leak-aware split and resized dataset
Run top to bottom, in the same Colab session as Day 2 if you can (it needs `day2_clusters.csv` and `day2_pairs.csv`; if they're not on the runtime it asks you to upload them).

**How the split works**
1. Every Day 2 cluster stays together: all images of a cluster go to the same split (train / val / test, about 80/10/10 per class).
2. Clusters can't catch every look-alike pair, so there is a second step, the **purge**: any val or test image that still has a similar partner (similarity at or above `PURGE_T`) in train is **dropped** from the dataset, and any test image similar to a kept val image is dropped too. Dropped images are not used anywhere. After this, no flagged pair crosses a split boundary.
3. You choose `PURGE_T` after seeing what each choice costs (step 5).
4. Images are resized once (short side 384 px) into `train/val/test/<class>/` folders so training is fast.

## 1. Setup, GPU check, load Day 2 results

In [ ]:
import os, glob, re, zipfile, subprocess, shutil
import numpy as np, pandas as pd
from PIL import Image

DATA = "/content/maize_data"
OUT = "/content/day2_out"
os.makedirs(OUT, exist_ok=True)

try:
    import torch
    print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
    else:
        print("No GPU now. Not needed for this notebook, but switch to a T4 (Runtime > Change runtime type) before Day 5 training.")
except ImportError:
    print("torch not installed")

need = {"clusters": "day2_clusters.csv", "pairs": "day2_pairs.csv"}
if not all(os.path.exists(f"{OUT}/{f}") for f in need.values()):
    from google.colab import files
    print("Upload day2_clusters.csv and day2_pairs.csv")
    for name, blob in files.upload().items():
        key = "clusters" if "cluster" in name else "pairs" if "pair" in name else None
        if key:
            open(f"{OUT}/{need[key]}", "wb").write(blob)

if not os.path.isdir(DATA):       # runtime reset: fetch the images again
    os.makedirs("/content/maize_zip", exist_ok=True)
    zp = "/content/maize_zip/maize.zip"
    subprocess.run(["wget", "-q", "-O", zp, "https://data.mendeley.com/public-api/zip/fkw49mz3xs/download/1"], check=True)
    os.makedirs(DATA, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(DATA)
    for _ in range(3):
        inner = glob.glob(DATA + "/**/*.zip", recursive=True)
        if not inner:
            break
        for p in inner:
            with zipfile.ZipFile(p) as z:
                z.extractall(p[:-4])
            os.remove(p)

df = pd.read_csv(f"{OUT}/day2_clusters.csv")     # row order = image index used by the pairs file
P = pd.read_csv(f"{OUT}/day2_pairs.csv")
df["abs_path"] = DATA + df["path"]
assert P.i.max() < len(df) and P.j.max() < len(df), "pairs file does not match clusters file (different runs?)"
assert os.path.exists(df["abs_path"].iloc[0]) and os.path.exists(df["abs_path"].iloc[-1]), "image paths not found under DATA"
print(len(df), "images,", len(P), "similar pairs (>=", round(P.sim.min(), 2), ")")
print(df["cls"].value_counts().to_dict())


## 2. Do neighbouring file numbers mean the same leaf? (diagnostic)
In the cross-class pairs you sent, MLN_2239, 2240 and 2241 looked like shots of the same leaf. If similar pairs usually have close file numbers, bursts are numbered consecutively, which is useful to know for the write-up. This cell only reports; it doesn't change the split.

In [ ]:
num = df["name"].str.extract(r"_(\d+)(?:_\d+)?$")[0].astype(float).values
same_cls = df["cls"].values[P.i.values] == df["cls"].values[P.j.values]
Pw = P[same_cls].copy()
Pw["gap"] = np.abs(num[Pw.j.values] - num[Pw.i.values])
rows = []
for lo in [0.85, 0.90, 0.94]:
    q = Pw[Pw.sim >= lo]
    rows.append({"min similarity": lo, "same-class pairs": len(q),
                 "file numbers within 3": f"{100 * (q.gap <= 3).mean():.0f}%" if len(q) else "-",
                 "within 10": f"{100 * (q.gap <= 10).mean():.0f}%" if len(q) else "-"})
print(pd.DataFrame(rows).to_string(index=False))


## 3. Split whole clusters, stratified by class

In [ ]:
SEED = 0
FRACS = {"train": 0.8, "val": 0.1, "test": 0.1}
rng = np.random.default_rng(SEED)

g = df.groupby("cluster").agg(size=("cluster", "size"), cls=("cls", lambda s: s.value_counts().index[0])).reset_index()
assign = {}
for c in sorted(df["cls"].unique()):
    gc = g[g.cls == c].copy()
    gc["r"] = rng.random(len(gc))
    gc = gc.sort_values(["size", "r"], ascending=[False, True])     # big clusters first
    total = int(gc["size"].sum())
    target = {s: f * total for s, f in FRACS.items()}
    have = {s: 0 for s in FRACS}
    for _, row in gc.iterrows():
        s = max(FRACS, key=lambda k: target[k] - have[k])            # split furthest below its target
        assign[row["cluster"]] = s
        have[s] += int(row["size"])
df["split"] = df["cluster"].map(assign)
print("after cluster split:")
print(pd.crosstab(df["cls"], df["split"])[["train", "val", "test"]])


## 4. Purge: drop val/test images that still have a similar partner in train

In [ ]:
def purge(T):
    """Return (val images to drop, test images to drop) for similarity cutoff T."""
    Q = P[P.sim >= T]
    a, b = Q.i.values, Q.j.values
    s = df["split"].values
    sa, sb = s[a], s[b]
    def partners(x, y):                      # images in split x with a partner in split y
        m1 = (sa == x) & (sb == y)
        m2 = (sb == x) & (sa == y)
        return np.concatenate([a[m1], b[m2]]), np.concatenate([b[m1], a[m2]])
    dv, _ = partners("val", "train")
    drop_val = set(dv.tolist())
    dt1, _ = partners("test", "train")
    dt2, pv = partners("test", "val")
    keep = np.array([p not in drop_val for p in pv], dtype=bool)
    drop_test = set(dt1.tolist()) | set(dt2[keep].tolist())
    return drop_val, drop_test

n_val, n_test = int((df.split == "val").sum()), int((df.split == "test").sum())
rows = []
for T in [0.85, 0.88, 0.90, 0.92, 0.94]:
    dv, dt = purge(T)
    rows.append({"PURGE_T": T, "val dropped": len(dv), "% of val": round(100 * len(dv) / n_val, 1),
                 "test dropped": len(dt), "% of test": round(100 * len(dt) / n_test, 1)})
print(pd.DataFrame(rows).to_string(index=False))


## 5. Choose `PURGE_T`
Lower is safer against leakage but drops more of val/test, and the images that get dropped are exactly the ones that resemble training images, so a heavy purge makes the test set a bit harder and less typical. That is the honest direction to err in, but you shouldn't throw away half your test set.

Rule of thumb: take the **lowest** `PURGE_T` that still leaves you comfortable (say, under about 20% dropped from val and test). Your cross-class pairs showed that different leaves can score 0.91 to 0.93, so cutoffs above that stop being meaningful as "same photo" anyway. Write your choice and the table above into your notes; they go in the write-up.

In [ ]:
PURGE_T = 0.90   # <-- SET THIS after reading the table above (this default is a guess)

dv, dt = purge(PURGE_T)
df["final_split"] = df["split"]
df.loc[list(dv) + list(dt), "final_split"] = "dropped"
fs = df["final_split"].values

def crossing(T):
    Q = P[P.sim >= T]
    a, b = Q.i.values, Q.j.values
    return int(((fs[a] != "dropped") & (fs[b] != "dropped") & (fs[a] != fs[b])).sum())

viol = crossing(PURGE_T)
print(f"similar pairs (>= {PURGE_T}) still crossing a split boundary: {viol}")
assert viol == 0
for T in sorted({0.85, 0.90} - {PURGE_T}):
    if T < PURGE_T:
        print(f"for information, pairs between {T} and {PURGE_T} still crossing: {crossing(T) - viol}")

tab = pd.crosstab(df["cls"], df["final_split"])
tab = tab[[c for c in ["train", "val", "test", "dropped"] if c in tab.columns]]
print("\nfinal split:")
print(tab)
print("\nclass shares within each split (%):")
print((pd.crosstab(df["cls"], df["final_split"], normalize="columns") * 100).round(1)[["train", "val", "test"]])


## 6. Write the resized dataset
Copies every kept image, resized so the short side is 384 px (never enlarged), into `/content/maize_split/<split>/<class>/`. Takes several minutes; safe to re-run (it skips files that exist).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
SHORT = 384
DST = "/content/maize_split"

keep = df[df["final_split"] != "dropped"].copy()
keep["dst"] = [f"{DST}/{s}/{c}/{n}.jpg" for s, c, n in zip(keep["final_split"], keep["cls"], keep["name"])]

def make(args):
    src, dst = args
    if os.path.exists(dst):
        return 0
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with Image.open(src) as im:
        im.draft("RGB", (SHORT * 2, SHORT * 2))
        im = im.convert("RGB")
        w, h = im.size
        s = SHORT / min(w, h)
        if s < 1:
            im = im.resize((round(w * s), round(h * s)), Image.LANCZOS)
        im.save(dst, quality=92)
    return 1

with ThreadPoolExecutor(max_workers=4) as ex:
    done = sum(ex.map(make, zip(keep["abs_path"], keep["dst"])))
print("written:", done, "| folders:")
for s in ["train", "val", "test"]:
    print(" ", s, {c: len(os.listdir(f"{DST}/{s}/{c}")) for c in sorted(os.listdir(f"{DST}/{s}"))})


## 7. Save the manifest and summary
`day3_split.csv` records exactly which image is in which split (keep it: it is your reproducibility record). Paste `day3_summary.md` back into the chat.

In [ ]:
man = df.drop(columns=["abs_path"])
man.to_csv(OUT + "/day3_split.csv", index=False)
n = len(df)
cnt = df["final_split"].value_counts()
lines = [
    "# Day 3 summary",
    f"- images: {n}; split by cluster (seed {SEED}), then purged at similarity {PURGE_T}",
    f"- train {cnt.get('train', 0)}, val {cnt.get('val', 0)}, test {cnt.get('test', 0)}, dropped {cnt.get('dropped', 0)} ({100 * cnt.get('dropped', 0) / n:.1f}%)",
    f"- dropped from val: {len(dv)} ({100 * len(dv) / n_val:.1f}% of val); from test: {len(dt)} ({100 * len(dt) / n_test:.1f}% of test)",
    f"- flagged pairs crossing a split after purge: {viol}",
    "- class counts per split:",
]
for c, r in tab.iterrows():
    lines.append(f"  - {c}: " + ", ".join(f"{k} {int(v)}" for k, v in r.items()))
open(OUT + "/day3_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
try:
    from google.colab import files
    files.download(OUT + "/day3_split.csv")
    files.download(OUT + "/day3_summary.md")
except ImportError:
    pass


## 8. (Optional) Keep the dataset for later sessions
Colab wipes `/content` when the runtime resets. Zipping the resized dataset (roughly half a GB) to Drive means Day 4 onward can skip all of this. Uncomment and run.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.make_archive("/content/drive/MyDrive/maize_split", "zip", DST)
# print("saved to Drive: maize_split.zip")
